# SpaceCore Speed Comparisons

This notebook compares bare backend operations with equivalent SpaceCore abstractions. Timing rows report steady-state runtime per call; JAX JIT rows compile during warmup and exclude compilation from the measured loop.


## What Is Compared

| Case | Bare operation | SpaceCore abstraction |
| --- | --- | --- |
| Dense forward | `A @ x` | `DenseLinOp(A, X, Y).apply(x)` |
| PyTorch eager | `A_t @ x_t` | `DenseLinOp(A_t, X_t, Y_t).apply(x_t)` |
| Dense adjoint | `AH @ y`, where `AH = A.T.conj()` is prepared once | `DenseLinOp(A, X, Y).rapply(y)` |
| Tensor-shaped dense forward | `(A2 @ x.ravel()).reshape(cod_shape)` | `DenseLinOp(A_tensor, X, Y).apply(x_tensor)` |
| Tensor-shaped dense adjoint | `(A2H @ y.ravel()).reshape(dom_shape)`, where `A2H = A2.T.conj()` is prepared once | `DenseLinOp(A_tensor, X, Y).rapply(y_tensor)` |
| Sparse forward | `S @ x` | `SparseLinOp(S, X, Y).apply(x)` |
| Sparse adjoint | `(ST @ y.conj()).conj()`, where `ST = S.T` is prepared once | `SparseLinOp(S, X, Y).rapply(y)` |
| Batched dense forward | `xs @ A.T` | `DenseLinOp(A, X, Y).vapply(xs)` |
| Batched dense adjoint | `ys @ AH.T` | `DenseLinOp(A, X, Y).rvapply(ys)` |
| Batched sparse forward | `(S @ xs.T).T` | `SparseLinOp(S, X, Y).vapply(xs)` |
| Batched sparse adjoint | `(ST @ ys.T).T` | `SparseLinOp(S, X, Y).rvapply(ys)` |
| Product flatten | `np.concatenate((x1.ravel(), x2.ravel()))` | `ProductSpace((X1, X2)).flatten((x1, x2))` |
| Product unflatten | array slicing plus reshape | `ProductSpace((X1, X2)).unflatten(v)` |
| Block diagonal forward | `(A1 @ x1, A2 @ x2)` | `BlockDiagonalLinOp.from_operators(...).apply((x1, x2))` |
| Sum-to-single forward | `B1 @ x1 + B2 @ x2` | `SumToSingleLinOp.from_operators(...).apply((x1, x2))` |
| Stacked forward | `(B1 @ x, B2 @ x)` | `StackedLinOp.from_operators(...).apply(x)` |
| Stacked adjoint | `B1H @ y1 + B2H @ y2`, where adjoints are prepared once | `StackedLinOp.from_operators(...).rapply((y1, y2))` |
| JAX JIT dense/product cases | `jax.jit(lambda ...: bare_backend_expression)` | `jax.jit(lambda ...: spacecore_object.method(...))` |

Bare operations include the reshapes or tuple assembly needed to return the same representation as the abstraction. JAX benchmark calls are synchronized with `block_until_ready()` so asynchronous dispatch does not hide runtime.


In [1]:
from __future__ import annotations

import gc
import html
import sys
from pathlib import Path
from time import perf_counter_ns

import numpy as np
import scipy.sparse as sps

ROOT = Path.cwd()
if not (ROOT / "spacecore").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import spacecore as sc  # noqa: E402

try:
    import jax

    jax.config.update("jax_enable_x64", True)
    import jax.numpy as jnp

    JAX_AVAILABLE = True
    JAX_IMPORT_ERROR = None
except Exception as exc:
    jax = None
    jnp = None
    JAX_AVAILABLE = False
    JAX_IMPORT_ERROR = exc

try:
    import torch

    TORCH_AVAILABLE = True
    TORCH_IMPORT_ERROR = None
except Exception as exc:
    torch = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = exc

rng = np.random.default_rng(0)
ctx = sc.Context(sc.NumpyOps(), dtype=np.float64, enable_checks=False)

TIMING_ROWS = []
RATIO_ROWS = []


def prod(shape):
    out = 1
    for dim in shape:
        out *= int(dim)
    return out


def wait_ready(value):
    if TORCH_AVAILABLE and isinstance(value, torch.Tensor):
        if value.is_cuda:
            torch.cuda.synchronize(value.device)
    elif hasattr(value, "block_until_ready"):
        value.block_until_ready()
    elif isinstance(value, (tuple, list)):
        for item in value:
            wait_ready(item)
    elif isinstance(value, dict):
        for item in value.values():
            wait_ready(item)
    return value


def bench(label, fn, *, repeat=40, number=1000, warmup=1):
    for _ in range(warmup):
        wait_ready(fn())
    samples = []
    for _ in range(repeat):
        was_enabled = gc.isenabled()
        gc.disable()
        try:
            start = perf_counter_ns()
            for _ in range(number):
                wait_ready(fn())
            elapsed = perf_counter_ns() - start
        finally:
            if was_enabled:
                gc.enable()
        samples.append(elapsed / number)
    return {
        "label": label,
        "best_us": min(samples) / 1_000,
        "median_us": float(np.median(samples)) / 1_000,
    }


def _timing_row(case, mode, kind, timing):
    return {
        "case": case,
        "mode": mode,
        "kind": kind,
        "expression": timing["label"],
        "best_us": timing["best_us"],
        "median_us": timing["median_us"],
    }


def ratio_rows(case, mode, bare, abstract):
    return {
        "case": case,
        "mode": mode,
        "bare": bare["label"],
        "abstraction": abstract["label"],
        "median_overhead_x": abstract["median_us"] / bare["median_us"],
        "best_overhead_x": abstract["best_us"] / bare["best_us"],
    }


def bench_pair(case, mode, bare_label, bare_fn, abstraction_label, abstraction_fn, *, repeat=200, number=5000, warmup=1):
    bare = bench(bare_label, bare_fn, repeat=repeat, number=number, warmup=warmup)
    abstraction = bench(abstraction_label, abstraction_fn, repeat=repeat, number=number, warmup=warmup)
    rows = [
        _timing_row(case, mode, "Bare", bare),
        _timing_row(case, mode, "SpaceCore", abstraction),
    ]
    ratios = [ratio_rows(case, mode, bare, abstraction)]
    TIMING_ROWS.extend(rows)
    RATIO_ROWS.extend(ratios)
    return rows, ratios


def add_pair(target_rows, target_ratios, *args, **kwargs):
    rows, ratios = bench_pair(*args, **kwargs)
    target_rows.extend(rows)
    target_ratios.extend(ratios)


def _fmt(value):
    if isinstance(value, float):
        return f"{value:.3f}"
    return str(value)


def _table_html(rows, columns):
    head = "".join(f"<th>{html.escape(col)}</th>" for col in columns)
    body_rows = []
    for row in rows:
        cells = "".join(f"<td>{html.escape(_fmt(row.get(col, '')))}</td>" for col in columns)
        body_rows.append(f"<tr>{cells}</tr>")
    return f"""
    <table class="speed-table">
      <thead><tr>{head}</tr></thead>
      <tbody>{''.join(body_rows)}</tbody>
    </table>
    """


def _timing_bars_html(rows):
    if not rows:
        return ""
    max_value = max(row["median_us"] for row in rows) or 1.0
    bars = []
    for row in rows:
        width = max(2.0, 100.0 * row["median_us"] / max_value)
        color = "#4c78a8" if row["kind"] == "Bare" else "#f58518"
        label = f"{row['mode']} / {row['case']} / {row['kind']}"
        bars.append(f"""
        <div class="speed-bar-row">
          <div class="speed-bar-label">{html.escape(label)}</div>
          <div class="speed-bar-track"><div class="speed-bar" style="width:{width:.2f}%; background:{color};"></div></div>
          <div class="speed-bar-value">{row['median_us']:.3f} us</div>
        </div>
        """)
    return "<h4>Median Time Per Call</h4>" + "".join(bars)


def _ratio_bars_html(ratios):
    if not ratios:
        return ""
    max_ratio = max(max(row["median_overhead_x"], 1.0) for row in ratios) or 1.0
    scale = min(max(max_ratio, 1.25), 3.0)
    bars = []
    for row in ratios:
        ratio = row["median_overhead_x"]
        width = max(2.0, 100.0 * min(ratio, scale) / scale)
        color = "#2f9e44" if ratio <= 1.05 else "#f08c00" if ratio <= 1.25 else "#c92a2a"
        label = f"{row['mode']} / {row['case']}"
        bars.append(f"""
        <div class="speed-bar-row">
          <div class="speed-bar-label">{html.escape(label)}</div>
          <div class="speed-bar-track ratio"><div class="speed-bar" style="width:{width:.2f}%; background:{color};"></div></div>
          <div class="speed-bar-value">{ratio:.3f}x</div>
        </div>
        """)
    return "<h4>SpaceCore / Bare Median Ratio</h4>" + "".join(bars)


def _display_html(content):
    try:
        from IPython.display import HTML, display

        display(HTML(content))
    except Exception:
        print(content)


def show_section(rows, ratios, *, title=None):
    title_html = f"<h3>{html.escape(title)}</h3>" if title else ""
    css = """
    <style>
      .speed-wrap { font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; }
      .speed-table { border-collapse: collapse; width: 100%; margin: 0.5rem 0 1rem; font-size: 13px; }
      .speed-table th, .speed-table td { border: 1px solid #d9dee7; padding: 6px 8px; text-align: left; vertical-align: top; }
      .speed-table th { background: #f4f6f8; font-weight: 650; }
      .speed-bar-row { display: grid; grid-template-columns: minmax(220px, 34%) 1fr 80px; gap: 10px; align-items: center; margin: 5px 0; font-size: 13px; }
      .speed-bar-label { overflow-wrap: anywhere; color: #1f2933; }
      .speed-bar-track { height: 12px; background: #edf1f5; border-radius: 3px; overflow: hidden; }
      .speed-bar-track.ratio { background: linear-gradient(90deg, #e9f7ef 0, #e9f7ef 40%, #fff4e6 40%, #fff4e6 55%, #fdecea 55%, #fdecea 100%); }
      .speed-bar { height: 100%; border-radius: 3px; }
      .speed-bar-value { font-variant-numeric: tabular-nums; text-align: right; color: #25313d; }
      .speed-note { color: #586574; font-size: 12px; margin: 0.25rem 0 1rem; }
      .speed-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap: 18px; }
    </style>
    """
    rows_table = _table_html(rows, ["mode", "case", "kind", "expression", "best_us", "median_us"])
    ratios_table = _table_html(ratios, ["mode", "case", "median_overhead_x", "best_overhead_x"])
    content = f"""
    {css}
    <div class="speed-wrap">
      {title_html}
      <div class="speed-note">Lower is faster. Ratio is SpaceCore median time divided by bare median time.</div>
      {rows_table}
      {ratios_table}
      <div class="speed-grid">
        <div>{_timing_bars_html(rows)}</div>
        <div>{_ratio_bars_html(ratios)}</div>
      </div>
    </div>
    """
    _display_html(content)


def show_speed_dashboard(rows=None, ratios=None):
    rows = TIMING_ROWS if rows is None else rows
    ratios = RATIO_ROWS if ratios is None else ratios
    show_section(rows, ratios, title="All Speed Comparisons")


## Dense Matrix Operator

Compares raw NumPy matrix-vector multiply with `DenseLinOp.apply`, and a cached raw adjoint matrix multiply with `DenseLinOp.rapply`.


In [2]:
section_rows = []
section_ratios = []

m, n = 512, 512
A = ctx.asarray(rng.standard_normal((m, n)))
AH = A.T.conj()
x = ctx.asarray(rng.standard_normal(n))
y = ctx.asarray(rng.standard_normal(m))
X = sc.VectorSpace((n,), ctx)
Y = sc.VectorSpace((m,), ctx)
dense_op = sc.DenseLinOp(A, X, Y, ctx)

add_pair(
    section_rows,
    section_ratios,
    "Dense forward",
    "NumPy eager",
    "A @ x",
    lambda: A @ x,
    "dense_op.apply(x)",
    lambda: dense_op.apply(x),
    number=500,
)
add_pair(
    section_rows,
    section_ratios,
    "Dense adjoint",
    "NumPy eager",
    "AH @ y",
    lambda: AH @ y,
    "dense_op.rapply(y)",
    lambda: dense_op.rapply(y),
    number=500,
)

show_section(section_rows, section_ratios, title="Dense Matrix Operator")


## Tensor-Shaped Dense Operator

Compares the bare reshaped matrix operation with a `DenseLinOp` whose stored array has shape `cod.shape + dom.shape`.


In [3]:
section_rows = []
section_ratios = []

dom_shape = (16, 8)
cod_shape = (32, 16)
A_tensor = ctx.asarray(rng.standard_normal(cod_shape + dom_shape))
A2 = A_tensor.reshape((prod(cod_shape), prod(dom_shape)))
A2H = A2.T.conj()
x_tensor = ctx.asarray(rng.standard_normal(dom_shape))
y_tensor = ctx.asarray(rng.standard_normal(cod_shape))
Xt = sc.VectorSpace(dom_shape, ctx)
Yt = sc.VectorSpace(cod_shape, ctx)
tensor_op = sc.DenseLinOp(A_tensor, Xt, Yt, ctx)

add_pair(
    section_rows,
    section_ratios,
    "Tensor dense forward",
    "NumPy eager",
    "(A2 @ x_tensor.ravel()).reshape(cod_shape)",
    lambda: (A2 @ x_tensor.reshape((-1,))).reshape(cod_shape),
    "tensor_op.apply(x_tensor)",
    lambda: tensor_op.apply(x_tensor),
    number=500,
)
add_pair(
    section_rows,
    section_ratios,
    "Tensor dense adjoint",
    "NumPy eager",
    "(A2H @ y_tensor.ravel()).reshape(dom_shape)",
    lambda: (A2H @ y_tensor.reshape((-1,))).reshape(dom_shape),
    "tensor_op.rapply(y_tensor)",
    lambda: tensor_op.rapply(y_tensor),
    number=500,
)

show_section(section_rows, section_ratios, title="Tensor-Shaped Dense Operator")


## Sparse Matrix Operator

Compares SciPy CSR matrix-vector products with `SparseLinOp.apply` and `SparseLinOp.rapply`. The raw adjoint uses a cached transpose to match the abstraction's cached setup.


In [4]:
section_rows = []
section_ratios = []

m, n = 2048, 2048
S = sps.random(m, n, density=0.003, format="csr", random_state=0, dtype=np.float64)
ST = S.T
xs = ctx.asarray(rng.standard_normal(n))
ys = ctx.asarray(rng.standard_normal(m))
Xs = sc.VectorSpace((n,), ctx)
Ys = sc.VectorSpace((m,), ctx)
sparse_op = sc.SparseLinOp(S, Xs, Ys, ctx)

add_pair(
    section_rows,
    section_ratios,
    "Sparse forward",
    "NumPy eager",
    "S @ xs",
    lambda: S @ xs,
    "sparse_op.apply(xs)",
    lambda: sparse_op.apply(xs),
    number=1000,
)
add_pair(
    section_rows,
    section_ratios,
    "Sparse adjoint",
    "NumPy eager",
    "(ST @ ys.conj()).conj()",
    lambda: (ST @ ys.conj()).conj(),
    "sparse_op.rapply(ys)",
    lambda: sparse_op.rapply(ys),
    number=1000,
)

show_section(section_rows, section_ratios, title="Sparse Matrix Operator")


## Batched Linear Operators

Checks and benchmarks `vapply` and `rvapply` against equivalent dense and sparse multi-vector operations.


In [5]:
section_rows = []
section_ratios = []

batch_size = 16
m, n = 512, 512
A_batch = ctx.asarray(rng.standard_normal((m, n)))
AH_batch = A_batch.T.conj()
batch_xs = ctx.asarray(rng.standard_normal((batch_size, n)))
batch_ys = ctx.asarray(rng.standard_normal((batch_size, m)))
X_batch = sc.VectorSpace((n,), ctx)
Y_batch = sc.VectorSpace((m,), ctx)
XB = X_batch.batch(batch_shape=(batch_size,), batch_axes=(0,))
YB = Y_batch.batch(batch_shape=(batch_size,), batch_axes=(0,))
dense_batch_op = sc.DenseLinOp(A_batch, X_batch, Y_batch, ctx)

S_batch = sps.random(m, n, density=0.003, format="csr", random_state=1, dtype=np.float64)
ST_batch = S_batch.T
sparse_batch_op = sc.SparseLinOp(S_batch, X_batch, Y_batch, ctx)

dense_v_ref = batch_xs @ A_batch.T
dense_rv_ref = batch_ys @ AH_batch.T
sparse_v_ref = (S_batch @ batch_xs.T).T
sparse_rv_ref = (ST_batch @ batch_ys.T).T

assert np.allclose(dense_batch_op.vapply(batch_xs, XB), dense_v_ref)
assert np.allclose(dense_batch_op.rvapply(batch_ys, YB), dense_rv_ref)
assert np.allclose(sparse_batch_op.vapply(batch_xs, XB), sparse_v_ref)
assert np.allclose(sparse_batch_op.rvapply(batch_ys, YB), sparse_rv_ref)

add_pair(
    section_rows,
    section_ratios,
    "Batched dense forward",
    "NumPy eager",
    "batch_xs @ A_batch.T",
    lambda: batch_xs @ A_batch.T,
    "dense_batch_op.vapply(batch_xs, XB)",
    lambda: dense_batch_op.vapply(batch_xs, XB),
    number=200,
)
add_pair(
    section_rows,
    section_ratios,
    "Batched dense adjoint",
    "NumPy eager",
    "batch_ys @ AH_batch.T",
    lambda: batch_ys @ AH_batch.T,
    "dense_batch_op.rvapply(batch_ys, YB)",
    lambda: dense_batch_op.rvapply(batch_ys, YB),
    number=200,
)
add_pair(
    section_rows,
    section_ratios,
    "Batched sparse forward",
    "NumPy eager",
    "(S_batch @ batch_xs.T).T",
    lambda: (S_batch @ batch_xs.T).T,
    "sparse_batch_op.vapply(batch_xs, XB)",
    lambda: sparse_batch_op.vapply(batch_xs, XB),
    number=500,
)
add_pair(
    section_rows,
    section_ratios,
    "Batched sparse adjoint",
    "NumPy eager",
    "(ST_batch @ batch_ys.T).T",
    lambda: (ST_batch @ batch_ys.T).T,
    "sparse_batch_op.rvapply(batch_ys, YB)",
    lambda: sparse_batch_op.rvapply(batch_ys, YB),
    number=500,
)

show_section(section_rows, section_ratios, title="Batched Linear Operators")


## Product Space Coordinates

Compares bare tuple flattening/unflattening with `ProductSpace.flatten` and `ProductSpace.unflatten`.


In [6]:
section_rows = []
section_ratios = []

shape1 = (64, 64)
shape2 = (2048,)
X1 = sc.VectorSpace(shape1, ctx)
X2 = sc.VectorSpace(shape2, ctx)
P = sc.ProductSpace((X1, X2), ctx)
x1 = ctx.asarray(rng.standard_normal(shape1))
x2 = ctx.asarray(rng.standard_normal(shape2))
v = ctx.asarray(rng.standard_normal(P.shape))
split = prod(shape1)

add_pair(
    section_rows,
    section_ratios,
    "Product flatten",
    "NumPy eager",
    "np.concatenate((x1.ravel(), x2.ravel()))",
    lambda: np.concatenate((x1.reshape((-1,)), x2.reshape((-1,))), axis=0),
    "P.flatten((x1, x2))",
    lambda: P.flatten((x1, x2)),
    number=1000,
)
add_pair(
    section_rows,
    section_ratios,
    "Product unflatten",
    "NumPy eager",
    "(v[:split].reshape(shape1), v[split:].reshape(shape2))",
    lambda: (v[:split].reshape(shape1), v[split:].reshape(shape2)),
    "P.unflatten(v)",
    lambda: P.unflatten(v),
    number=1000,
)

show_section(section_rows, section_ratios, title="Product Space Coordinates")


## Product Linear Operators

Compares hand-written tuple operations with `BlockDiagonalLinOp`, `SumToSingleLinOp`, and `StackedLinOp`.


In [7]:
section_rows = []
section_ratios = []

n1, n2, m1, m2 = 256, 128, 256, 128
A1 = ctx.asarray(rng.standard_normal((m1, n1)))
A2 = ctx.asarray(rng.standard_normal((m2, n2)))
x1 = ctx.asarray(rng.standard_normal(n1))
x2 = ctx.asarray(rng.standard_normal(n2))
X1 = sc.VectorSpace((n1,), ctx)
X2 = sc.VectorSpace((n2,), ctx)
Y1 = sc.VectorSpace((m1,), ctx)
Y2 = sc.VectorSpace((m2,), ctx)
op1 = sc.DenseLinOp(A1, X1, Y1, ctx)
op2 = sc.DenseLinOp(A2, X2, Y2, ctx)
block = sc.BlockDiagonalLinOp.from_operators((op1, op2))

add_pair(
    section_rows,
    section_ratios,
    "Block diagonal forward",
    "NumPy eager",
    "(A1 @ x1, A2 @ x2)",
    lambda: (A1 @ x1, A2 @ x2),
    "block.apply((x1, x2))",
    lambda: block.apply((x1, x2)),
    number=500,
)

n_common = 256
m_common = 256
B1 = ctx.asarray(rng.standard_normal((m_common, n_common)))
B2 = ctx.asarray(rng.standard_normal((m_common, n_common)))
B1H = B1.T.conj()
B2H = B2.T.conj()
xb1 = ctx.asarray(rng.standard_normal(n_common))
xb2 = ctx.asarray(rng.standard_normal(n_common))
yb = ctx.asarray(rng.standard_normal(m_common))
X_common = sc.VectorSpace((n_common,), ctx)
Y_common = sc.VectorSpace((m_common,), ctx)
sum_op = sc.SumToSingleLinOp.from_operators((
    sc.DenseLinOp(B1, X_common, Y_common, ctx),
    sc.DenseLinOp(B2, X_common, Y_common, ctx),
))
stacked_op = sc.StackedLinOp.from_operators((
    sc.DenseLinOp(B1, X_common, Y_common, ctx),
    sc.DenseLinOp(B2, X_common, Y_common, ctx),
))

add_pair(
    section_rows,
    section_ratios,
    "Sum-to-single forward",
    "NumPy eager",
    "B1 @ xb1 + B2 @ xb2",
    lambda: B1 @ xb1 + B2 @ xb2,
    "sum_op.apply((xb1, xb2))",
    lambda: sum_op.apply((xb1, xb2)),
    number=500,
)
add_pair(
    section_rows,
    section_ratios,
    "Stacked forward",
    "NumPy eager",
    "(B1 @ xb1, B2 @ xb1)",
    lambda: (B1 @ xb1, B2 @ xb1),
    "stacked_op.apply(xb1)",
    lambda: stacked_op.apply(xb1),
    number=500,
)
add_pair(
    section_rows,
    section_ratios,
    "Stacked adjoint",
    "NumPy eager",
    "B1H @ yb + B2H @ yb",
    lambda: B1H @ yb + B2H @ yb,
    "stacked_op.rapply((yb, yb))",
    lambda: stacked_op.rapply((yb, yb)),
    number=500,
)

show_section(section_rows, section_ratios, title="Product Linear Operators")


## PyTorch Eager Benchmarks

These benchmarks compare raw PyTorch tensor operations with the same operations routed through SpaceCore abstractions. They run on PyTorch's default device for the created tensors and skip if PyTorch is not installed.

In [8]:
section_rows = []
section_ratios = []

if not TORCH_AVAILABLE:
    print(f"Skipping PyTorch eager benchmarks: {TORCH_IMPORT_ERROR!r}")
else:
    tctx = sc.Context(sc.TorchOps(), dtype=torch.float64, enable_checks=False)

    tm, tn = 512, 512
    At = tctx.asarray(rng.standard_normal((tm, tn)))
    AHt = At.T.conj()
    xt = tctx.asarray(rng.standard_normal(tn))
    yt = tctx.asarray(rng.standard_normal(tm))
    Xt = sc.VectorSpace((tn,), tctx)
    Yt = sc.VectorSpace((tm,), tctx)
    dense_top = sc.DenseLinOp(At, Xt, Yt, tctx)

    add_pair(
        section_rows,
        section_ratios,
        "Dense forward",
        "PyTorch eager",
        "At @ xt",
        lambda: At @ xt,
        "dense_top.apply(xt)",
        lambda: dense_top.apply(xt),
        number=500,
    )
    add_pair(
        section_rows,
        section_ratios,
        "Dense adjoint",
        "PyTorch eager",
        "AHt @ yt",
        lambda: AHt @ yt,
        "dense_top.rapply(yt)",
        lambda: dense_top.rapply(yt),
        number=500,
    )

    tdom_shape = (16, 8)
    tcod_shape = (32, 16)
    At_tensor = tctx.asarray(rng.standard_normal(tcod_shape + tdom_shape))
    At2 = At_tensor.reshape((prod(tcod_shape), prod(tdom_shape)))
    At2H = At2.T.conj()
    xt_tensor = tctx.asarray(rng.standard_normal(tdom_shape))
    yt_tensor = tctx.asarray(rng.standard_normal(tcod_shape))
    Xtt = sc.VectorSpace(tdom_shape, tctx)
    Ytt = sc.VectorSpace(tcod_shape, tctx)
    tensor_top = sc.DenseLinOp(At_tensor, Xtt, Ytt, tctx)

    add_pair(
        section_rows,
        section_ratios,
        "Tensor dense forward",
        "PyTorch eager",
        "(At2 @ xt_tensor.ravel()).reshape(tcod_shape)",
        lambda: (At2 @ xt_tensor.reshape((-1,))).reshape(tcod_shape),
        "tensor_top.apply(xt_tensor)",
        lambda: tensor_top.apply(xt_tensor),
        number=500,
    )
    add_pair(
        section_rows,
        section_ratios,
        "Tensor dense adjoint",
        "PyTorch eager",
        "(At2H @ yt_tensor.ravel()).reshape(tdom_shape)",
        lambda: (At2H @ yt_tensor.reshape((-1,))).reshape(tdom_shape),
        "tensor_top.rapply(yt_tensor)",
        lambda: tensor_top.rapply(yt_tensor),
        number=500,
    )

    tbatch_size = 16
    tbatch_xs = tctx.asarray(rng.standard_normal((tbatch_size, tn)))
    tbatch_ys = tctx.asarray(rng.standard_normal((tbatch_size, tm)))
    tXB = Xt.batch(batch_shape=(tbatch_size,), batch_axes=(0,))
    tYB = Yt.batch(batch_shape=(tbatch_size,), batch_axes=(0,))

    assert torch.allclose(dense_top.vapply(tbatch_xs, tXB), tbatch_xs @ At.T)
    assert torch.allclose(dense_top.rvapply(tbatch_ys, tYB), tbatch_ys @ AHt.T)

    add_pair(
        section_rows,
        section_ratios,
        "Batched dense forward",
        "PyTorch eager",
        "tbatch_xs @ At.T",
        lambda: tbatch_xs @ At.T,
        "dense_top.vapply(tbatch_xs, tXB)",
        lambda: dense_top.vapply(tbatch_xs, tXB),
        number=200,
    )
    add_pair(
        section_rows,
        section_ratios,
        "Batched dense adjoint",
        "PyTorch eager",
        "tbatch_ys @ AHt.T",
        lambda: tbatch_ys @ AHt.T,
        "dense_top.rvapply(tbatch_ys, tYB)",
        lambda: dense_top.rvapply(tbatch_ys, tYB),
        number=200,
    )

    try:
        ts_m, ts_n = 1024, 1024
        TS_source = sps.random(ts_m, ts_n, density=0.003, format="csr", random_state=2, dtype=np.float64)
        TS = tctx.assparse(TS_source)
        TST = TS.T
        ts_xs = tctx.asarray(rng.standard_normal((tbatch_size, ts_n)))
        ts_ys = tctx.asarray(rng.standard_normal((tbatch_size, ts_m)))
        tXs = sc.VectorSpace((ts_n,), tctx)
        tYs = sc.VectorSpace((ts_m,), tctx)
        tXBs = tXs.batch(batch_shape=(tbatch_size,), batch_axes=(0,))
        tYBs = tYs.batch(batch_shape=(tbatch_size,), batch_axes=(0,))
        tsparse_op = sc.SparseLinOp(TS, tXs, tYs, tctx)

        assert torch.allclose(tsparse_op.vapply(ts_xs, tXBs), torch.sparse.mm(TS, ts_xs.T).T)
        assert torch.allclose(tsparse_op.rvapply(ts_ys, tYBs), torch.sparse.mm(TST, ts_ys.T).T)

        add_pair(
            section_rows,
            section_ratios,
            "Batched sparse forward",
            "PyTorch eager",
            "torch.sparse.mm(TS, ts_xs.T).T",
            lambda: torch.sparse.mm(TS, ts_xs.T).T,
            "tsparse_op.vapply(ts_xs, tXBs)",
            lambda: tsparse_op.vapply(ts_xs, tXBs),
            number=200,
        )
        add_pair(
            section_rows,
            section_ratios,
            "Batched sparse adjoint",
            "PyTorch eager",
            "torch.sparse.mm(TST, ts_ys.T).T",
            lambda: torch.sparse.mm(TST, ts_ys.T).T,
            "tsparse_op.rvapply(ts_ys, tYBs)",
            lambda: tsparse_op.rvapply(ts_ys, tYBs),
            number=200,
        )
    except Exception as exc:
        print(f"Skipping PyTorch sparse batched benchmarks: {exc!r}")

    tshape1 = (64, 64)
    tshape2 = (2048,)
    tX1 = sc.VectorSpace(tshape1, tctx)
    tX2 = sc.VectorSpace(tshape2, tctx)
    tP = sc.ProductSpace((tX1, tX2), tctx)
    tx1 = tctx.asarray(rng.standard_normal(tshape1))
    tx2 = tctx.asarray(rng.standard_normal(tshape2))
    tv = tctx.asarray(rng.standard_normal(tP.shape))
    tsplit = prod(tshape1)

    add_pair(
        section_rows,
        section_ratios,
        "Product flatten",
        "PyTorch eager",
        "torch.cat((tx1.ravel(), tx2.ravel()))",
        lambda: torch.cat((tx1.reshape((-1,)), tx2.reshape((-1,))), dim=0),
        "tP.flatten((tx1, tx2))",
        lambda: tP.flatten((tx1, tx2)),
        number=1000,
    )
    add_pair(
        section_rows,
        section_ratios,
        "Product unflatten",
        "PyTorch eager",
        "(tv[:tsplit].reshape(tshape1), tv[tsplit:].reshape(tshape2))",
        lambda: (tv[:tsplit].reshape(tshape1), tv[tsplit:].reshape(tshape2)),
        "tP.unflatten(tv)",
        lambda: tP.unflatten(tv),
        number=1000,
    )

show_section(section_rows, section_ratios, title="PyTorch Eager Benchmarks")

/Users/pavlopelikh/Documents/Dev/SpaceCore/spacecore/backend/torch/_ops.py:239: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  out = self.torch.sparse_coo_tensor(indices, values, coo.shape, device=device)


mode,case,kind,expression,best_us,median_us
PyTorch eager,Dense forward,Bare,At @ xt,9.717,10.701
PyTorch eager,Dense forward,SpaceCore,dense_top.apply(xt),10.240,11.573
PyTorch eager,Dense adjoint,Bare,AHt @ yt,7.309,7.626
PyTorch eager,Dense adjoint,SpaceCore,dense_top.rapply(yt),7.514,7.673
PyTorch eager,Tensor dense forward,Bare,(At2 @ xt_tensor.ravel()).reshape(tcod_shape),7.420,7.676
PyTorch eager,Tensor dense forward,SpaceCore,tensor_top.apply(xt_tensor),7.640,8.064
PyTorch eager,Tensor dense adjoint,Bare,(At2H @ yt_tensor.ravel()).reshape(tdom_shape),5.242,5.524
PyTorch eager,Tensor dense adjoint,SpaceCore,tensor_top.rapply(yt_tensor),5.522,5.737
PyTorch eager,Batched dense forward,Bare,tbatch_xs @ At.T,54.294,56.286
PyTorch eager,Batched dense forward,SpaceCore,"dense_top.vapply(tbatch_xs, tXB)",55.530,59.023


## JAX JIT-Compiled Benchmarks

These benchmarks compare already-compiled bare JAX functions with already-compiled functions that call SpaceCore methods. The first warmup call triggers compilation and is not included in the measured samples.


In [9]:
section_rows = []
section_ratios = []

if not JAX_AVAILABLE:
    print(f"Skipping JAX JIT benchmarks: {JAX_IMPORT_ERROR!r}")
else:
    jctx = sc.Context(sc.JaxOps(), dtype=jnp.float64, enable_checks=False)

    jm, jn = 512, 512
    Aj = jctx.asarray(rng.standard_normal((jm, jn)))
    AHj = Aj.T.conj()
    xj = jctx.asarray(rng.standard_normal(jn))
    yj = jctx.asarray(rng.standard_normal(jm))
    Xj = sc.VectorSpace((jn,), jctx)
    Yj = sc.VectorSpace((jm,), jctx)
    dense_jop = sc.DenseLinOp(Aj, Xj, Yj, jctx)

    dense_forward_bare_jit = jax.jit(lambda z: Aj @ z)
    dense_forward_abs_jit = jax.jit(lambda z: dense_jop.apply(z))
    dense_adjoint_bare_jit = jax.jit(lambda z: AHj @ z)
    dense_adjoint_abs_jit = jax.jit(lambda z: dense_jop.rapply(z))

    add_pair(
        section_rows,
        section_ratios,
        "Dense forward",
        "JAX JIT",
        "jax.jit(lambda z: Aj @ z)(xj)",
        lambda: dense_forward_bare_jit(xj),
        "jax.jit(lambda z: dense_jop.apply(z))(xj)",
        lambda: dense_forward_abs_jit(xj),
        number=500,
        warmup=2,
    )
    add_pair(
        section_rows,
        section_ratios,
        "Dense adjoint",
        "JAX JIT",
        "jax.jit(lambda z: AHj @ z)(yj)",
        lambda: dense_adjoint_bare_jit(yj),
        "jax.jit(lambda z: dense_jop.rapply(z))(yj)",
        lambda: dense_adjoint_abs_jit(yj),
        number=500,
        warmup=2,
    )

    jdom_shape = (16, 8)
    jcod_shape = (32, 16)
    Aj_tensor = jctx.asarray(rng.standard_normal(jcod_shape + jdom_shape))
    Aj2 = Aj_tensor.reshape((prod(jcod_shape), prod(jdom_shape)))
    Aj2H = Aj2.T.conj()
    xj_tensor = jctx.asarray(rng.standard_normal(jdom_shape))
    yj_tensor = jctx.asarray(rng.standard_normal(jcod_shape))
    Xjt = sc.VectorSpace(jdom_shape, jctx)
    Yjt = sc.VectorSpace(jcod_shape, jctx)
    tensor_jop = sc.DenseLinOp(Aj_tensor, Xjt, Yjt, jctx)

    tensor_forward_bare_jit = jax.jit(lambda z: (Aj2 @ z.reshape((-1,))).reshape(jcod_shape))
    tensor_forward_abs_jit = jax.jit(lambda z: tensor_jop.apply(z))
    tensor_adjoint_bare_jit = jax.jit(lambda z: (Aj2H @ z.reshape((-1,))).reshape(jdom_shape))
    tensor_adjoint_abs_jit = jax.jit(lambda z: tensor_jop.rapply(z))

    add_pair(
        section_rows,
        section_ratios,
        "Tensor dense forward",
        "JAX JIT",
        "jax.jit(lambda z: (Aj2 @ z.ravel()).reshape(jcod_shape))(xj_tensor)",
        lambda: tensor_forward_bare_jit(xj_tensor),
        "jax.jit(lambda z: tensor_jop.apply(z))(xj_tensor)",
        lambda: tensor_forward_abs_jit(xj_tensor),
        number=500,
        warmup=2,
    )
    add_pair(
        section_rows,
        section_ratios,
        "Tensor dense adjoint",
        "JAX JIT",
        "jax.jit(lambda z: (Aj2H @ z.ravel()).reshape(jdom_shape))(yj_tensor)",
        lambda: tensor_adjoint_bare_jit(yj_tensor),
        "jax.jit(lambda z: tensor_jop.rapply(z))(yj_tensor)",
        lambda: tensor_adjoint_abs_jit(yj_tensor),
        number=500,
        warmup=2,
    )

    jbatch_size = 16
    jbatch_xs = jctx.asarray(rng.standard_normal((jbatch_size, jn)))
    jbatch_ys = jctx.asarray(rng.standard_normal((jbatch_size, jm)))
    jXB = Xj.batch(batch_shape=(jbatch_size,), batch_axes=(0,))
    jYB = Yj.batch(batch_shape=(jbatch_size,), batch_axes=(0,))

    dense_batched_forward_bare_jit = jax.jit(lambda z: z @ Aj.T)
    dense_batched_forward_abs_jit = jax.jit(lambda z: dense_jop.vapply(z, jXB))
    dense_batched_adjoint_bare_jit = jax.jit(lambda z: z @ AHj.T)
    dense_batched_adjoint_abs_jit = jax.jit(lambda z: dense_jop.rvapply(z, jYB))

    assert jnp.allclose(dense_batched_forward_abs_jit(jbatch_xs), dense_batched_forward_bare_jit(jbatch_xs))
    assert jnp.allclose(dense_batched_adjoint_abs_jit(jbatch_ys), dense_batched_adjoint_bare_jit(jbatch_ys))

    add_pair(
        section_rows,
        section_ratios,
        "Batched dense forward",
        "JAX JIT",
        "jax.jit(lambda z: z @ Aj.T)(jbatch_xs)",
        lambda: dense_batched_forward_bare_jit(jbatch_xs),
        "jax.jit(lambda z: dense_jop.vapply(z, jXB))(jbatch_xs)",
        lambda: dense_batched_forward_abs_jit(jbatch_xs),
        number=200,
        warmup=2,
    )
    add_pair(
        section_rows,
        section_ratios,
        "Batched dense adjoint",
        "JAX JIT",
        "jax.jit(lambda z: z @ AHj.T)(jbatch_ys)",
        lambda: dense_batched_adjoint_bare_jit(jbatch_ys),
        "jax.jit(lambda z: dense_jop.rvapply(z, jYB))(jbatch_ys)",
        lambda: dense_batched_adjoint_abs_jit(jbatch_ys),
        number=200,
        warmup=2,
    )

    jshape1 = (64, 64)
    jshape2 = (2048,)
    jX1 = sc.VectorSpace(jshape1, jctx)
    jX2 = sc.VectorSpace(jshape2, jctx)
    jP = sc.ProductSpace((jX1, jX2), jctx)
    jx1 = jctx.asarray(rng.standard_normal(jshape1))
    jx2 = jctx.asarray(rng.standard_normal(jshape2))
    jv = jctx.asarray(rng.standard_normal(jP.shape))
    jsplit = prod(jshape1)

    product_flat_bare_jit = jax.jit(lambda a, b: jnp.concatenate((a.reshape((-1,)), b.reshape((-1,))), axis=0))
    product_flat_abs_jit = jax.jit(lambda a, b: jP.flatten((a, b)))
    product_unflat_bare_jit = jax.jit(lambda z: (z[:jsplit].reshape(jshape1), z[jsplit:].reshape(jshape2)))
    product_unflat_abs_jit = jax.jit(lambda z: jP.unflatten(z))

    add_pair(
        section_rows,
        section_ratios,
        "Product flatten",
        "JAX JIT",
        "jax.jit(lambda a, b: jnp.concatenate((a.ravel(), b.ravel())))(jx1, jx2)",
        lambda: product_flat_bare_jit(jx1, jx2),
        "jax.jit(lambda a, b: jP.flatten((a, b)))(jx1, jx2)",
        lambda: product_flat_abs_jit(jx1, jx2),
        number=1000,
        warmup=2,
    )
    add_pair(
        section_rows,
        section_ratios,
        "Product unflatten",
        "JAX JIT",
        "jax.jit(lambda z: (z[:split].reshape(shape1), z[split:].reshape(shape2)))(jv)",
        lambda: product_unflat_bare_jit(jv),
        "jax.jit(lambda z: jP.unflatten(z))(jv)",
        lambda: product_unflat_abs_jit(jv),
        number=1000,
        warmup=2,
    )

    jn1, jn2, jm1, jm2 = 256, 128, 256, 128
    JA1 = jctx.asarray(rng.standard_normal((jm1, jn1)))
    JA2 = jctx.asarray(rng.standard_normal((jm2, jn2)))
    jpx1 = jctx.asarray(rng.standard_normal(jn1))
    jpx2 = jctx.asarray(rng.standard_normal(jn2))
    jPX1 = sc.VectorSpace((jn1,), jctx)
    jPX2 = sc.VectorSpace((jn2,), jctx)
    jPY1 = sc.VectorSpace((jm1,), jctx)
    jPY2 = sc.VectorSpace((jm2,), jctx)
    jop1 = sc.DenseLinOp(JA1, jPX1, jPY1, jctx)
    jop2 = sc.DenseLinOp(JA2, jPX2, jPY2, jctx)
    jblock = sc.BlockDiagonalLinOp.from_operators((jop1, jop2))

    block_bare_jit = jax.jit(lambda a, b: (JA1 @ a, JA2 @ b))
    block_abs_jit = jax.jit(lambda a, b: jblock.apply((a, b)))

    add_pair(
        section_rows,
        section_ratios,
        "Block diagonal forward",
        "JAX JIT",
        "jax.jit(lambda a, b: (JA1 @ a, JA2 @ b))(jpx1, jpx2)",
        lambda: block_bare_jit(jpx1, jpx2),
        "jax.jit(lambda a, b: jblock.apply((a, b)))(jpx1, jpx2)",
        lambda: block_abs_jit(jpx1, jpx2),
        number=500,
        warmup=2,
    )

    jn_common = 256
    jm_common = 256
    JB1 = jctx.asarray(rng.standard_normal((jm_common, jn_common)))
    JB2 = jctx.asarray(rng.standard_normal((jm_common, jn_common)))
    JB1H = JB1.T.conj()
    JB2H = JB2.T.conj()
    jxb1 = jctx.asarray(rng.standard_normal(jn_common))
    jxb2 = jctx.asarray(rng.standard_normal(jn_common))
    jyb = jctx.asarray(rng.standard_normal(jm_common))
    jX_common = sc.VectorSpace((jn_common,), jctx)
    jY_common = sc.VectorSpace((jm_common,), jctx)
    jsum_op = sc.SumToSingleLinOp.from_operators((
        sc.DenseLinOp(JB1, jX_common, jY_common, jctx),
        sc.DenseLinOp(JB2, jX_common, jY_common, jctx),
    ))
    jstacked_op = sc.StackedLinOp.from_operators((
        sc.DenseLinOp(JB1, jX_common, jY_common, jctx),
        sc.DenseLinOp(JB2, jX_common, jY_common, jctx),
    ))

    sum_bare_jit = jax.jit(lambda a, b: JB1 @ a + JB2 @ b)
    sum_abs_jit = jax.jit(lambda a, b: jsum_op.apply((a, b)))
    stack_bare_jit = jax.jit(lambda z: (JB1 @ z, JB2 @ z))
    stack_abs_jit = jax.jit(lambda z: jstacked_op.apply(z))
    stack_adj_bare_jit = jax.jit(lambda a, b: JB1H @ a + JB2H @ b)
    stack_adj_abs_jit = jax.jit(lambda a, b: jstacked_op.rapply((a, b)))

    add_pair(
        section_rows,
        section_ratios,
        "Sum-to-single forward",
        "JAX JIT",
        "jax.jit(lambda a, b: JB1 @ a + JB2 @ b)(jxb1, jxb2)",
        lambda: sum_bare_jit(jxb1, jxb2),
        "jax.jit(lambda a, b: jsum_op.apply((a, b)))(jxb1, jxb2)",
        lambda: sum_abs_jit(jxb1, jxb2),
        number=500,
        warmup=2,
    )
    add_pair(
        section_rows,
        section_ratios,
        "Stacked forward",
        "JAX JIT",
        "jax.jit(lambda z: (JB1 @ z, JB2 @ z))(jxb1)",
        lambda: stack_bare_jit(jxb1),
        "jax.jit(lambda z: jstacked_op.apply(z))(jxb1)",
        lambda: stack_abs_jit(jxb1),
        number=500,
        warmup=2,
    )
    add_pair(
        section_rows,
        section_ratios,
        "Stacked adjoint",
        "JAX JIT",
        "jax.jit(lambda a, b: JB1H @ a + JB2H @ b)(jyb, jyb)",
        lambda: stack_adj_bare_jit(jyb, jyb),
        "jax.jit(lambda a, b: jstacked_op.rapply((a, b)))(jyb, jyb)",
        lambda: stack_adj_abs_jit(jyb, jyb),
        number=500,
        warmup=2,
    )

    try:
        js_m, js_n = 2048, 2048
        JS = jctx.assparse(sps.random(js_m, js_n, density=0.003, format="csr", random_state=1, dtype=np.float64))
        JST = JS.T
        jxs = jctx.asarray(rng.standard_normal(js_n))
        jys = jctx.asarray(rng.standard_normal(js_m))
        jXs = sc.VectorSpace((js_n,), jctx)
        jYs = sc.VectorSpace((js_m,), jctx)
        jsparse_op = sc.SparseLinOp(JS, jXs, jYs, jctx)

        sparse_forward_bare_jit = jax.jit(lambda z: JS @ z)
        sparse_forward_abs_jit = jax.jit(lambda z: jsparse_op.apply(z))
        sparse_adjoint_bare_jit = jax.jit(lambda z: (JST @ z.conj()).conj())
        sparse_adjoint_abs_jit = jax.jit(lambda z: jsparse_op.rapply(z))

        add_pair(
            section_rows,
            section_ratios,
            "Sparse forward",
            "JAX JIT",
            "jax.jit(lambda z: JS @ z)(jxs)",
            lambda: sparse_forward_bare_jit(jxs),
            "jax.jit(lambda z: jsparse_op.apply(z))(jxs)",
            lambda: sparse_forward_abs_jit(jxs),
            number=500,
            warmup=2,
        )
        add_pair(
            section_rows,
            section_ratios,
            "Sparse adjoint",
            "JAX JIT",
            "jax.jit(lambda z: (JST @ z.conj()).conj())(jys)",
            lambda: sparse_adjoint_bare_jit(jys),
            "jax.jit(lambda z: jsparse_op.rapply(z))(jys)",
            lambda: sparse_adjoint_abs_jit(jys),
            number=500,
            warmup=2,
        )


        jbatch_sparse_xs = jctx.asarray(rng.standard_normal((jbatch_size, js_n)))
        jbatch_sparse_ys = jctx.asarray(rng.standard_normal((jbatch_size, js_m)))
        jXBs = jXs.batch(batch_shape=(jbatch_size,), batch_axes=(0,))
        jYBs = jYs.batch(batch_shape=(jbatch_size,), batch_axes=(0,))

        sparse_batched_forward_bare_jit = jax.jit(lambda z: (JS @ z.T).T)
        sparse_batched_forward_abs_jit = jax.jit(lambda z: jsparse_op.vapply(z, jXBs))
        sparse_batched_adjoint_bare_jit = jax.jit(lambda z: (JST @ z.T).T)
        sparse_batched_adjoint_abs_jit = jax.jit(lambda z: jsparse_op.rvapply(z, jYBs))

        assert jnp.allclose(
            sparse_batched_forward_abs_jit(jbatch_sparse_xs),
            sparse_batched_forward_bare_jit(jbatch_sparse_xs),
        )
        assert jnp.allclose(
            sparse_batched_adjoint_abs_jit(jbatch_sparse_ys),
            sparse_batched_adjoint_bare_jit(jbatch_sparse_ys),
        )

        add_pair(
            section_rows,
            section_ratios,
            "Batched sparse forward",
            "JAX JIT",
            "jax.jit(lambda z: (JS @ z.T).T)(jbatch_sparse_xs)",
            lambda: sparse_batched_forward_bare_jit(jbatch_sparse_xs),
            "jax.jit(lambda z: jsparse_op.vapply(z, jXBs))(jbatch_sparse_xs)",
            lambda: sparse_batched_forward_abs_jit(jbatch_sparse_xs),
            number=200,
            warmup=2,
        )
        add_pair(
            section_rows,
            section_ratios,
            "Batched sparse adjoint",
            "JAX JIT",
            "jax.jit(lambda z: (JST @ z.T).T)(jbatch_sparse_ys)",
            lambda: sparse_batched_adjoint_bare_jit(jbatch_sparse_ys),
            "jax.jit(lambda z: jsparse_op.rvapply(z, jYBs))(jbatch_sparse_ys)",
            lambda: sparse_batched_adjoint_abs_jit(jbatch_sparse_ys),
            number=200,
            warmup=2,
        )
    except Exception as exc:
        print(f"Skipping sparse JIT benchmarks: {exc!r}")

show_section(section_rows, section_ratios, title="JAX JIT-Compiled Benchmarks")


mode,case,kind,expression,best_us,median_us
JAX JIT,Dense forward,Bare,jax.jit(lambda z: Aj @ z)(xj),40.272,48.793
JAX JIT,Dense forward,SpaceCore,jax.jit(lambda z: dense_jop.apply(z))(xj),40.894,48.053
JAX JIT,Dense adjoint,Bare,jax.jit(lambda z: AHj @ z)(yj),38.618,46.501
JAX JIT,Dense adjoint,SpaceCore,jax.jit(lambda z: dense_jop.rapply(z))(yj),41.546,45.345
JAX JIT,Tensor dense forward,Bare,jax.jit(lambda z: (Aj2 @ z.ravel()).reshape(jcod_shape))(xj_tensor),15.397,26.639
JAX JIT,Tensor dense forward,SpaceCore,jax.jit(lambda z: tensor_jop.apply(z))(xj_tensor),16.407,24.959
JAX JIT,Tensor dense adjoint,Bare,jax.jit(lambda z: (Aj2H @ z.ravel()).reshape(jdom_shape))(yj_tensor),18.525,22.117
JAX JIT,Tensor dense adjoint,SpaceCore,jax.jit(lambda z: tensor_jop.rapply(z))(yj_tensor),20.235,22.564
JAX JIT,Batched dense forward,Bare,jax.jit(lambda z: z @ Aj.T)(jbatch_xs),208.200,233.987
JAX JIT,Batched dense forward,SpaceCore,"jax.jit(lambda z: dense_jop.vapply(z, jXB))(jbatch_xs)",210.622,231.717


## Speed Dashboard

This combines every benchmark row produced above into one table and two bar charts.


In [10]:
show_speed_dashboard()


mode,case,kind,expression,best_us,median_us
NumPy eager,Dense forward,Bare,A @ x,9.731,10.048
NumPy eager,Dense forward,SpaceCore,dense_op.apply(x),9.982,10.231
NumPy eager,Dense adjoint,Bare,AH @ y,6.957,7.282
NumPy eager,Dense adjoint,SpaceCore,dense_op.rapply(y),7.389,7.544
NumPy eager,Tensor dense forward,Bare,(A2 @ x_tensor.ravel()).reshape(cod_shape),5.009,5.148
NumPy eager,Tensor dense forward,SpaceCore,tensor_op.apply(x_tensor),5.279,5.388
NumPy eager,Tensor dense adjoint,Bare,(A2H @ y_tensor.ravel()).reshape(dom_shape),2.720,2.759
NumPy eager,Tensor dense adjoint,SpaceCore,tensor_op.rapply(y_tensor),2.988,3.046
NumPy eager,Sparse forward,Bare,S @ xs,8.649,9.392
NumPy eager,Sparse forward,SpaceCore,sparse_op.apply(xs),9.001,10.063


## Measuring Validation Cost

The cells above use `enable_checks=False` so they isolate dispatch and representation overhead. To measure validation overhead separately, change the context creation in the setup cell to:

```python
ctx = sc.Context(sc.NumpyOps(), dtype=np.float64, enable_checks=True)
```

For JAX benchmarks, make the same change in the `jctx = sc.Context(...)` line. Then rerun the notebook. That compares bare linear algebra with SpaceCore abstractions plus membership checks.
